# Preprocessing dữ liệu Summarization tiếng Việt

Notebook này chạy pipeline preprocessing cho project `NLP-Abstractive-Summary`.

Input là hai file parquet trong thư mục `data/`:

- `data/train-00000-of-00001.parquet`
- `data/valid-00000-of-00001.parquet`

Output clean sẽ được lưu vào `data/processed/`. File model-ready cuối cùng chỉ giữ hai cột `article` và `summary` để các phần training đọc schema đơn giản.

Notebook này dùng lại logic trong `src/preprocessing_pipeline.py`; không sửa `src/preprocess.py` vì file đó đang phục vụ tokenizer/vocabulary.

## 1. Setup đường dẫn và import

Cell này tìm project root bằng đường dẫn tương đối. Vì vậy collaborator có thể chạy notebook từ project root hoặc từ thư mục `notebooks/` mà không cần sửa đường dẫn theo máy cá nhân.

In [1]:
from pathlib import Path
import sys

# Khi chạy bằng Python thường trên Windows, stdout có thể không phải UTF-8.
# Dòng này giúp print tiếng Việt có dấu không bị lỗi encoding.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

import pandas as pd

# Dùng display trong Jupyter; fallback sang print nếu chạy code ngoài notebook.
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

# Tìm project root bằng đường dẫn tương đối, không hard-code ổ đĩa hay username.
root_candidates = [Path("."), Path("..")]
PROJECT_ROOT = next(
    (
        path
        for path in root_candidates
        if (path / "src" / "preprocessing_pipeline.py").exists()
        and (path / "data" / "train-00000-of-00001.parquet").exists()
        and (path / "data" / "valid-00000-of-00001.parquet").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy project root. Hãy chạy notebook từ project root hoặc thư mục notebooks/."
    )

project_root_text = str(PROJECT_ROOT)
if project_root_text not in sys.path:
    sys.path.insert(0, project_root_text)

from src.preprocessing_pipeline import PreprocessConfig, run_preprocessing

print("Project root tương đối:", PROJECT_ROOT)

Project root tương đối: ..


## 2. Cấu hình pipeline

Các path dưới đây đều là path tương đối từ project root. Output được gom vào `data/processed/` để tách dữ liệu raw và dữ liệu đã xử lý.

In [2]:
config = PreprocessConfig(
    project_root=PROJECT_ROOT,
    train_file="data/train-00000-of-00001.parquet",
    valid_file="data/valid-00000-of-00001.parquet",
    output_dir="data/processed",
)

print("Train input:", config.train_path)
print("Valid input:", config.valid_path)
print("Processed dir:", config.processed_dir)
print("Report dir:", config.report_dir)
print("Figure dir:", config.figure_dir)

Train input: ..\data\train-00000-of-00001.parquet
Valid input: ..\data\valid-00000-of-00001.parquet
Processed dir: ..\data\processed
Report dir: ..\data\processed\reports
Figure dir: ..\data\processed\figures


## 3. Chạy EDA, cleaning và lưu output

Pipeline thực hiện các nhóm việc chính:

- Đọc hai split parquet và kiểm tra hai cột bắt buộc `article`, `summary`.
- Tạo report EDA trước cleaning: null, rỗng, duplicate, pattern kỹ thuật, độ dài, compression ratio.
- Làm sạch lỗi kỹ thuật: HTML entity/tag, Unicode NFC, ký tự điều khiển, whitespace, quote/dash, URL/email.
- Xóa các lỗi chắc chắn: article rỗng, summary rỗng, article giống summary, duplicate pair.
- Gắn warning flag vào DataFrame audit để review dữ liệu nghi vấn.
- Kiểm tra leakage train-valid theo `article`; nếu có overlap thì xóa ở train và giữ valid.
- Lưu parquet clean, report CSV và figure vào `data/processed/`.

In [3]:
# write_outputs=True nghĩa là notebook sẽ ghi file parquet/report/figure vào data/processed/.
# make_plot_files=True sẽ sinh histogram trong data/processed/figures/.
result = run_preprocessing(
    config=config,
    write_outputs=True,
    make_plot_files=True,
)

train_model = result["train_model"]
valid_model = result["valid_model"]

print("Train clean shape:", train_model.shape)
print("Valid clean shape:", valid_model.shape)
display(train_model.head(3))
display(valid_model.head(3))

Train clean shape: (10775, 2)
Valid clean shape: (1349, 2)


,article,summary
0,Gần 20 sự kiện được tổ chức trên toàn thành ph...,Hà Nội tổ chức gần 20 sự kiện từ 19/4 đến 10/5...
1,"Được thành lập năm 1897 tại Đức, Kempinski Hot...",Kempinski Hotels là một thương hiệu nổi tiếng ...
2,"Ngoài di chuyển đến Tuần Châu bằng đường bộ, m...",Bài viết giới thiệu các hoạt động vui chơi giả...


,article,summary
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ..."


## 4. Warning flags được gắn ở đâu?

Warning flags là các cột boolean được thêm tạm vào DataFrame audit sau cleaning, ví dụ:

- `warn_short_article`
- `warn_short_summary`
- `warn_summary_longer`
- `warn_high_compression`
- `warn_low_compression`

Các cột này **không dùng để xóa dữ liệu**. Chúng chỉ giúp review các dòng nghi vấn, ví dụ summary quá ngắn hoặc compression ratio quá cao.

Quan trọng: parquet clean dùng cho training chỉ giữ `article`, `summary`. Warning flags không đi vào model-ready data.

In [4]:
# train_clean / valid_clean là DataFrame audit: có thêm feature độ dài, cột warn_* và metadata.
# train_model / valid_model là DataFrame model-ready: chỉ có article và summary.
train_clean_audit = result["train_clean"]
valid_clean_audit = result["valid_clean"]

warning_cols = [col for col in train_clean_audit.columns if col.startswith("warn_")]

print("Các cột warning trong audit DataFrame:", warning_cols)
print("Schema train_model dùng cho training:", train_model.columns.tolist())

# Bảng này là summary số dòng bị gắn từng warning flag theo split.
# Nó được lưu ra data/processed/reports/warning_flags_summary.csv.
display(result["warning_report"])

# Xem thử vài dòng có warning nếu tồn tại.
warn_mask = train_clean_audit[warning_cols].any(axis=1) if warning_cols else pd.Series(False, index=train_clean_audit.index)
warning_preview_cols = ["source_index", "article_space_tokens", "summary_space_tokens", "compression_ratio_space"] + warning_cols
display(train_clean_audit.loc[warn_mask, warning_preview_cols].head(10))

Các cột warning trong audit DataFrame: ['warn_short_article', 'warn_short_summary', 'warn_summary_longer', 'warn_high_compression', 'warn_low_compression']
Schema train_model dùng cho training: ['article', 'summary']


,split,warn_short_article,warn_short_summary,warn_summary_longer,warn_high_compression,warn_low_compression
0,train,0,0,0,0,0
1,valid,0,0,0,0,0


,source_index,article_space_tokens,summary_space_tokens,compression_ratio_space,warn_short_article,warn_short_summary,warn_summary_longer,warn_high_compression,warn_low_compression


## 5. Đọc report CSV

Thư mục `data/processed/reports/` dùng để audit và giải thích dữ liệu sau preprocessing:

- `preprocess_overview_report.csv`: số dòng, null, rỗng, duplicate, `article == summary` trước/sau cleaning.
- `preprocess_quality_report.csv`: số dòng khớp các pattern kỹ thuật như URL, email, multi-space, HTML, control char.
- `accent_ratio_report.csv`: tỷ lệ ký tự có dấu tiếng Việt, chỉ để phát hiện dữ liệu nghi vấn.
- `length_percentiles_space_tokens.csv`: percentile độ dài article/summary và compression ratio.
- `warning_flags_summary.csv`: số dòng bị gắn từng warning flag.
- `leakage_report.csv`: overlap train-valid trước/sau xử lý leakage.
- `preprocess_removed_rows.csv`: log các dòng bị xóa và lý do xóa.

Các report này không phải input training; chúng dùng để kiểm tra chất lượng dữ liệu và giải thích quyết định preprocessing.

In [5]:
report_paths = result["output_paths"]

print("Các file output chính:")
for name, path in report_paths.items():
    print(f"{name}: {path}")

print("\nOverview report:")
display(result["overview_report"])

print("\nLength percentiles:")
display(result["length_report"])

print("\nLeakage report:")
display(result["leakage_report"])

print("\nRemoved rows sample:")
display(result["removed_rows"].head(10))

Các file output chính:
train_clean: ..\data\processed\train_clean.parquet
valid_clean: ..\data\processed\valid_clean.parquet
overview_report: ..\data\processed\reports\preprocess_overview_report.csv
quality_report: ..\data\processed\reports\preprocess_quality_report.csv
accent_report: ..\data\processed\reports\accent_ratio_report.csv
length_report: ..\data\processed\reports\length_percentiles_space_tokens.csv
warning_report: ..\data\processed\reports\warning_flags_summary.csv
leakage_report: ..\data\processed\reports\leakage_report.csv
removed_rows: ..\data\processed\reports\preprocess_removed_rows.csv

Overview report:


,stage,split,rows,columns,article_null,summary_null,article_empty,summary_empty,duplicate_rows,duplicate_pairs,article_equals_summary
0,raw,train,10775,2,0,0,0,0,0,0,0
1,raw,valid,1349,2,0,0,0,0,0,0,0
2,clean,train,10775,2,0,0,0,0,0,0,0
3,clean,valid,1349,2,0,0,0,0,0,0,0



Length percentiles:


,stage,split,metric,p50,p75,p90,p95,p99,max
0,raw,train,article_char_len,2061.000000,2402.500000,2854.600000,3042.000000,3232.000000,3580.000000
1,raw,train,summary_char_len,458.000000,540.000000,611.000000,645.000000,694.000000,792.000000
2,raw,train,article_space_tokens,444.000000,515.000000,615.000000,657.000000,691.000000,700.000000
3,raw,train,summary_space_tokens,100.000000,118.000000,134.000000,142.000000,149.000000,174.000000
4,raw,train,compression_ratio_space,0.225653,0.270270,0.312428,0.340265,0.391145,0.485050
5,raw,valid,article_char_len,2085.000000,2462.000000,2904.200000,3059.000000,3252.080000,3434.000000
6,raw,valid,summary_char_len,460.000000,536.000000,611.200000,643.600000,695.080000,785.000000
7,raw,valid,article_space_tokens,450.000000,533.000000,628.400000,663.600000,693.000000,700.000000
8,raw,valid,summary_space_tokens,100.000000,118.000000,134.000000,141.000000,148.000000,164.000000
9,raw,valid,compression_ratio_space,0.221918,0.263427,0.306204,0.331311,0.384947,0.486667



Leakage report:


,article_overlap_before,pair_overlap_before,train_rows_removed_by_article_leakage,valid_rows_removed_by_leakage,article_overlap_after,pair_overlap_after
0,0,0,0,0,0,0



Removed rows sample:


,article,summary,source_index,split,article_raw,summary_raw,article_char_len,summary_char_len,article_space_tokens,summary_space_tokens,compression_ratio_space,article_eq_summary,summary_longer_than_article,remove_reason,pair_key,warn_short_article,warn_short_summary,warn_summary_longer,warn_high_compression,warn_low_compression


## 6. Figure dùng để làm gì?

Thư mục `data/processed/figures/` lưu histogram phục vụ EDA:

- `train_article_space_tokens.png`: phân phối độ dài article.
- `train_summary_space_tokens.png`: phân phối độ dài summary.
- `train_compression_ratio.png`: phân phối tỷ lệ nén summary/article.

Các biểu đồ này giúp chọn `max_source_length`, `max_target_length` hoặc phát hiện outlier sơ bộ. Tuy nhiên độ dài ở đây là `space-token`, tức là tách bằng khoảng trắng. Với tiếng Việt, đây chỉ là ước lượng EDA, chưa thay thế tokenizer thật của model.

In [6]:
figure_paths = result["figure_paths"]["figures"]

print("Các figure đã lưu:")
for path in figure_paths:
    print("-", path)

# Nếu chạy trong Jupyter, có thể mở từng file PNG từ sidebar hoặc dùng cell hiển thị ảnh riêng.
# Ở đây chỉ kiểm tra file tồn tại để notebook không phụ thuộc thêm thư viện hiển thị ảnh.
for path in figure_paths:
    assert path.exists(), f"Thiếu figure: {path}"

Các figure đã lưu:
- ..\data\processed\figures\train_article_space_tokens.png
- ..\data\processed\figures\train_summary_space_tokens.png
- ..\data\processed\figures\train_compression_ratio.png


## 7. Acceptance checks

Các assert cuối notebook đảm bảo output phù hợp để dùng cho training: đúng schema, không null/rỗng, không duplicate pair, không `article == summary`, không còn overlap train-valid.

In [7]:
# Schema model-ready chỉ gồm hai cột chính.
assert list(train_model.columns) == ["article", "summary"]
assert list(valid_model.columns) == ["article", "summary"]

# Không còn null hoặc chuỗi rỗng trong hai cột chính.
for split_name, df in {"train": train_model, "valid": valid_model}.items():
    assert df[["article", "summary"]].isnull().sum().sum() == 0, f"{split_name} còn null"
    assert df["article"].astype(str).str.strip().ne("").all(), f"{split_name} còn article rỗng"
    assert df["summary"].astype(str).str.strip().ne("").all(), f"{split_name} còn summary rỗng"
    assert not df["article"].astype(str).str.strip().eq(df["summary"].astype(str).str.strip()).any(), f"{split_name} còn article == summary"
    pair_key = df["article"].astype(str) + " [SEP] " + df["summary"].astype(str)
    assert pair_key.duplicated().sum() == 0, f"{split_name} còn duplicate pair"

# Không còn article overlap giữa train và valid sau clean.
article_overlap = set(train_model["article"]).intersection(set(valid_model["article"]))
assert len(article_overlap) == 0, "Train-valid vẫn còn overlap theo article"

# Kiểm tra các output chính đã được ghi ra disk.
for path in report_paths.values():
    assert path.exists(), f"Thiếu output: {path}"
for path in figure_paths:
    assert path.exists(), f"Thiếu figure: {path}"

print("Tất cả acceptance checks đã pass.")

Tất cả acceptance checks đã pass.
